In [1]:
import parse_data as p_d
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

In [2]:
def add_grade_group(df, threshold=10):
    df = df.copy()
    df['grade_group'] = np.where(df['G3'] >= threshold, 'High', 'Low')
    return df

def fit_interaction_model(df, threshold=10):
    df2 = add_grade_group(df, threshold=threshold)

    model = smf.ols('G3 ~ absences * grade_group', data=df2).fit()
    return model

def summarize_interaction(model):

    params = model.params
    pvalues = model.pvalues

    interaction_name = [name for name in params.index if 'absences:grade_group' in name][0]

    beta_int = params[interaction_name]
    p_int = pvalues[interaction_name]

    print("Interaction term:", interaction_name)
    print(f"  beta = {beta_int:.3f}")
    print(f"  p    = {p_int:.4f}")


def separate_correlations(df, threshold=10):
    df2 = add_grade_group(df, threshold=threshold)

    corrs = {}
    for grp in ['Low', 'High']:
        sub = df2[df2['grade_group'] == grp]
        r, p = stats.pearsonr(sub['absences'], sub['G3'])
        corrs[grp] = {'r': r, 'p': p, 'n': len(sub)}

    return corrs

def one_way_anova_studytime(df, print_table=True):
    sub = df[['studytime', 'G3']].dropna()

    groups = {}
    for level in sorted(sub['studytime'].unique()):
        vals = sub.loc[sub['studytime'] == level, 'G3'].values
        groups[int(level)] = vals

    if print_table:
        print("studytime  n   mean(G3)   sd(G3)")
        for level, vals in groups.items():
            mean = np.mean(vals)
            sd   = np.std(vals, ddof=1)
            print(f"{level:9d} {len(vals):3d} {mean:9.3f} {sd:9.3f}")
        print()

    F, p = stats.f_oneway(*groups.values())

    if print_table:
        df_between = len(groups) - 1
        df_within  = len(sub) - len(groups)
        print(f"One-way ANOVA (studytime -> G3)")
        print(f"  F({df_between}, {df_within}) = {F:.3f},  p = {p:.4f}")

    return F, p

In [3]:
df = p_d.df

In [4]:
separate_correlations(df)

{'Low': {'r': np.float64(0.39566560868793366),
  'p': np.float64(3.1729329186683326e-06),
  'n': 130},
 'High': {'r': np.float64(-0.10190062272214852),
  'p': np.float64(0.09786532904154197),
  'n': 265}}

In [5]:
model = fit_interaction_model(df)
summarize_interaction(model)

Interaction term: absences:grade_group[T.Low]
  beta = 0.178
  p    = 0.0000


In [6]:
one_way_anova_studytime(df)

studytime  n   mean(G3)   sd(G3)
        1 105    10.048     4.956
        2 198    10.172     4.218
        3  65    11.400     4.640
        4  27    11.259     5.281

One-way ANOVA (studytime -> G3)
  F(3, 391) = 1.728,  p = 0.1607


(np.float64(1.7278351054436032), np.float64(0.16072280968365998))